**Daniel Lyu, Ariel Pan**

Spring 2026

CS 443: Bio-inspired Machine Learning

# Project 4: Recurrent Neural Networks

We will implement a character-level artificial RNN to explore how neural networks generate text when given a user prompt. Before building the network, let's focus on the text preprocessing pipeline. We will use the same IMDb movie review dataset from the word embedding project. Thus, the eventual text that you generate with the RNNs will resemble a "AI-generated" movie review! 

In [1]:
from text_dataset_char import CharLevelDataset

%load_ext autoreload
%autoreload 2

2026-05-08 00:56:54.052921: I tensorflow/core/platform/cpu_feature_guard.cc:211] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Task 0: Copy over files from previous project and download new data

Like with previous projects, we will reuse and build upon on our neural network library and preprocessing pipeline to save time. File to copy over from your most recent project:
- `layers.py`
- `network.py`
- `tf_util.py`
- `text_util.py`
- `skipgram_layers.py`

Additionally, download the following text datasets that we will use for this project:
- `imdb_raw.csv`
- `little_pigs.csv`

## Task 1: Text preprocessing for character-level models

The goal of this notebook is to write the text preprocessing pipeline for a **character-level model**. Unlike the SOM project where the Skipgram network processed and predicted *words*, this means that our recurrent neural network will process and predict text *single characters at a time*. 

Analogous to the `WordLevelDataset` class in `text_dataset_word.py` from the SOM project, in this project the `CharLevelDataset` class in `text_dataset_char.py` will handle preprocessing the dataset into character tokens. The `CharLevelDataset` class mirrors the structure and design of `WordLevelDataset` so it would likely be helpful to have that class open as you code.

### 1a. `CharLevelDataset` constructor, get, and `load` methods

Start by implementing:
1. the constructor
2. get methods
3. `load`

in the `CharLevelDataset` class. This should help get you acquainted with the instance variables in the class.

**Recall:** We will be using the following "invisible" ASCII non-printing chars in our character-level preprocessing pipeline:
- NUL character (`\x00`) for padding text sequences. We will split each movie review into a number fixed-length text sequences. We will be able to make that fixed length `seq_len` (**T**) whatever we want, but because reviews are different lengths and don't divide cleanly into lengths we pick, we need to **pad** the sequences to ensure we always have `seq_len` chars in each sequence. This pad char will be used for that and we will always pad the *end* of each review as needed.
- STX character (`\x02`) for marking/"tagging" the start of each review. This will allow the net to learn what the start of movie reviews "look like".
- ETX character (`\x03`) for marking/"tagging" the end of each review. This will allow the net to learn what the end of movie reviews "look like". This will also allow the net to decide when it should end its reviews on its own!

Let's verify the number of chars in the 1st 5 reviews

In [2]:
imdb_ds = CharLevelDataset(file_path='data/imdb_raw.csv')
corpus = imdb_ds.load(5)
review_num_chars = [len(review) for review in corpus]
print('The number of chars in each of the 1st 5 reviews is:')
print(review_num_chars)
print('and they should be:')
print([1725, 962, 902, 712, 1269])

The number of chars in each of the 1st 5 reviews is:
[1725, 962, 902, 712, 1269]
and they should be:
[1725, 962, 902, 712, 1269]


Let's examine the actual content of the 4th and 5th reviews (line wrapped to 100 chars).

In [3]:
for r, review in enumerate(corpus[-2:]):
    print(f'***Review {r+4}/{len(corpus)}:***')

    for c in range(0, len(review), 100):
        print(review[c:c+100])

***Review 4/5:***
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his par
ents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides 
to become Rambo and kill the zombie.OK, first of all when you're going to make a film you must Decid
e if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing l
ike in real life. And then we have Jake with his closet which totally ruins all the film! I expected
 to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spot
s.3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just 
ignore them.
***Review 5/5:***
Petter Mattei's "Love in the Time of Money" is a visually stunning film to watch. Mr. Mattei offers 
us a vivid portrait about human relations. This is a movie that seems to be telling us what money, p
ower and success do to people in the diffe

The output of cell above should be:

```
***Review 4/5:***
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his par
ents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides 
to become Rambo and kill the zombie.OK, first of all when you're going to make a film you must Decid
e if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing l
ike in real life. And then we have Jake with his closet which totally ruins all the film! I expected
 to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spot
s.3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just 
ignore them.
***Review 5/5:***
Petter Mattei's "Love in the Time of Money" is a visually stunning film to watch. Mr. Mattei offers 
us a vivid portrait about human relations. This is a movie that seems to be telling us what money, p
ower and success do to people in the different situations we encounter. This being a variation on th
e Arthur Schnitzler's play about the same theme, the director transfers the action to the present ti
me New York where all these different characters meet and connect. Each one is connected in one way,
 or another to the next person, but no one seems to know the previous point of contact. Stylishly, t
he film has a sophisticated luxurious look. We are taken to see how these people live and the world 
they live in their own habitat.The only thing one gets out of all these souls in the picture is the 
different stages of loneliness each one inhabits. A big city is not exactly the best place in which 
human relations find sincere fulfillment, as one discerns is the case with most of the people we enc
ounter.The acting is good under Mr. Mattei's direction. Steve Buscemi, Rosario Dawson, Carol Kane, M
ichael Imperioli, Adrian Grenier, and the rest of the talented cast, make these characters come aliv
e.We wish Mr. Mattei good luck and await anxiously for his next work.
```

### 1b. Create the vocabulary with the `make_vocabulary` method

Given that single characters are the "units" for a character-level model, the vocabulary represents all the unique characters in the corpus, along with the pad, start, and end tokens.

#### Test: vocabulary

In [4]:
vocab = imdb_ds.make_vocabulary(corpus)
print(f'Your vocab length is {len(vocab)} and it should be 67.')
print('Your vocab chars are:')
' '.join(vocab)

Your vocab length is 67 and it should be 67.
Your vocab chars are:


'\x00 \x02 \x03   ! " & \' ( ) , - . 0 1 2 3 : ? A B C D E F G H I J K L M N O P R S T W Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z'

The output of the cell above should be:

```
Your vocab chars are:

'\x00 \x02 \x03   ! " & \' ( ) , - . 0 1 2 3 : ? A B C D E F G H I J K L M N O P R S T W Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z'
```

### 1c. Create character-to-index and index-to-character maps

Like Skipgram, the recurrent neural network will need to process text in int-coded form. So we will need to convert the chars to an int code using a character-to-index map (dictionary). In order to interpret the int code, we will need to convert back to chars too with the index-to-character map (dictionary).

Implement this functionality in:
- `make_char2ind_mapping`
- `make_ind2char_mapping`

Test char ←→ int code mappings

In [5]:
char2ind_map = imdb_ds.make_char2ind_mapping(vocab)
ind2char_map = imdb_ds.make_ind2char_mapping(vocab)

# For this test, we sort your keys to make this consistent across teams
char2ind_map = dict(sorted(char2ind_map.items()))
ind2char_map = dict(sorted(ind2char_map.items()))

print(f'Your char2ind map has {len(char2ind_map)} entries and should have 67.')
print(f'Your ind2char_map map has {len(ind2char_map)} entries and should have 67.')

print('The 1st 5 char2ind entries are:')
for entry in list(char2ind_map.keys())[:5]:
    print(entry, '→', char2ind_map[entry])
print('The last 5 char2ind entries are:')
for entry in list(char2ind_map.keys())[-5:]:
    print(entry, '→', char2ind_map[entry])

print('The 1st 5 ind2char_map entries are:')
for entry in list(ind2char_map.keys())[:5]:
    print(entry, '→', ind2char_map[entry])
print('The last 5 ind2char_map entries are:')
for entry in list(ind2char_map.keys())[-5:]:
    print(entry, '→', ind2char_map[entry])

Your char2ind map has 67 entries and should have 67.
Your ind2char_map map has 67 entries and should have 67.
The 1st 5 char2ind entries are:
  → 0
 → 1
 → 2
  → 3
! → 4
The last 5 char2ind entries are:
v → 62
w → 63
x → 64
y → 65
z → 66
The 1st 5 ind2char_map entries are:
0 →  
1 → 
2 → 
3 →  
4 → !
The last 5 ind2char_map entries are:
62 → v
63 → w
64 → x
65 → y
66 → z


The last portion of the cell above should output:

*Note: The pad, start, and end tokens are "non-printing" chars so they should look "invisible"/whitespace when printed.*

```
The 1st 5 char2ind entries are:
 → 0
 → 1
 → 2
  → 3
! → 4
The last 5 char2ind entries are:
v → 62
w → 63
x → 64
y → 65
z → 66
The 1st 5 ind2char_map entries are:
0 → 
1 → 
2 → 
3 →  
4 → !
The last 5 ind2char_map entries are:
62 → v
63 → w
64 → x
65 → y
66 → z
```

### 1d. Divide corpus into fixed-length character sequences

Unlike Skipgram, which can form mini-batches of tokens in any order, the recurrent neural network needs to process text in ordered, sequences. So we will need to take each review and divide it up into sequences of length `seq_len` (e.g. 10 chars, 50 chars, 100 chars...whatever we want). By default, these sequences would be non-overlapping (i.e. `seq_overlap = 0`). For example:

> review = 'abcd'

If `seq_len = 2`, we have `seqs = ['ab', 'cd']`. However, it would be nice for the recurrent neural network to learn "smooth transitions" BETWEEN sequences, otherwise it will treat the two sequences as independent/not inheritly related — which is usually not the case (e.g. 2 ADJACENT sequences FLOW INTO EACH OTHER in of the SAME movie review, `'cd'` FOLLOWS `ab`). We can sidestep this issue by introducing **overlap** between successive sequences from the SAME review. For example:

> review = 'abcd'

If `seq_len = 2` and `seq_overlap = 1`, we have `seqs = ['ab', 'bc', 'cd']`. Now the net can "bridge the gap" across sequences with the shared chars!

Implement this process in the `make_sequences` method then test it below.

**Note:** The recurrent neural network is trained to predict the next char in each sequence. Thus, the labels/y is a sequence of chars (called **target sequences**), just like the **input sequences**, but are offset by 1 character. Your `make_sequences` should return these offset sequences as well.

Test `make_sequences`

In [6]:
from text_util import decode_special_tokens

Test making non-overlapping sequences of 1 review

In [7]:
# Focus on 2nd review in this test

seq_len = 53
seq_overlap = 0

seqs_x_str, seqs_y_str = imdb_ds.make_sequences(corpus=[corpus[1]], seq_len=seq_len, seq_overlap=seq_overlap)

print('2nd review:')
print('----------------')
for chunk in seqs_x_str:
    chunk = decode_special_tokens(chunk)
    print(chunk)
print('----------------')

2nd review:
----------------
<START>A wonderful little production. The filming technique
 is very unassuming- very old-time-BBC fashion and gi
ves a comforting, and sometimes discomforting, sense 
of realism to the entire piece. The actors are extrem
ely well chosen- Michael Sheen not only "has got all 
the polari" but he has all the voices down pat too! Y
ou can truly see the seamless editing guided by the r
eferences to Williams' diary entries, not only is it 
well worth the watching but it is a terrificly writte
n and performed piece. A masterful production about o
ne of the great master's of comedy and his life. The 
realism really comes home with the little things: the
 fantasy of the guard which, rather than use the trad
itional 'dream' techniques remains solid then disappe
ars. It plays on our knowledge and our senses, partic
ularly with the scenes concerning Orton and Halliwell
 and the sets (particularly of their flat with Halliw
ell's murals decorating every surface) are terr

The output of the cell above should be:

*Note: With the exception of the PAD token, pay attention that your lines break at exactly the correct chars.*

```
2nd review:
----------------
<START>A wonderful little production. The filming technique
 is very unassuming- very old-time-BBC fashion and gi
ves a comforting, and sometimes discomforting, sense 
of realism to the entire piece. The actors are extrem
ely well chosen- Michael Sheen not only "has got all 
the polari" but he has all the voices down pat too! Y
ou can truly see the seamless editing guided by the r
eferences to Williams' diary entries, not only is it 
well worth the watching but it is a terrificly writte
n and performed piece. A masterful production about o
ne of the great master's of comedy and his life. The 
realism really comes home with the little things: the
 fantasy of the guard which, rather than use the trad
itional 'dream' techniques remains solid then disappe
ars. It plays on our knowledge and our senses, partic
ularly with the scenes concerning Orton and Halliwell
 and the sets (particularly of their flat with Halliw
ell's murals decorating every surface) are terribly w
ell done.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
----------------
```

Test making non-overlapping sequences of 2 reviews

In [8]:
# Focus on 3rd and 4th reviews in this test

seq_len = 75
seq_overlap = 0

seqs_x_str, seqs_y_str = imdb_ds.make_sequences(corpus=corpus[2:4], seq_len=seq_len, seq_overlap=seq_overlap)

print('3rd and 4th reviews:')
print('----------------')
for chunk in seqs_x_str:
    chunk = decode_special_tokens(chunk)
    print(chunk)
print('----------------')

3rd and 4th reviews:
----------------
<START>I thought this was a wonderful way to spend time on a too hot summer weeke
nd, sitting in the air conditioned theater and watching a light-hearted com
edy. The plot is simplistic, but the dialogue is witty and the characters a
re likable (even the well bread suspected serial killer). While some may be
 disappointed when they realize this is not Match Point 2: Risk Addiction, 
I thought it was proof that Woody Allen is still fully in control of the st
yle many of us have grown to love.This was the most I'd laughed at one of W
oody's comedies in years (dare I say a decade?). While I've never been impr
essed with Scarlet Johanson, in this she managed to tone down her "sexy" im
age and jumped right into a average, but spirited young woman.This may not 
be the crown jewel of his career, but it was wittier than "Devil Wears Prad
a" and more interesting than "Superman" a great comedy to go see with frien
ds.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><

The output of the cell above should be:

*Note: With the exception of the PAD token, pay attention that your lines break at exactly the correct chars.*

```
3rd and 4th reviews:
----------------
<START>I thought this was a wonderful way to spend time on a too hot summer weeke
nd, sitting in the air conditioned theater and watching a light-hearted com
edy. The plot is simplistic, but the dialogue is witty and the characters a
re likable (even the well bread suspected serial killer). While some may be
 disappointed when they realize this is not Match Point 2: Risk Addiction, 
I thought it was proof that Woody Allen is still fully in control of the st
yle many of us have grown to love.This was the most I'd laughed at one of W
oody's comedies in years (dare I say a decade?). While I've never been impr
essed with Scarlet Johanson, in this she managed to tone down her "sexy" im
age and jumped right into a average, but spirited young woman.This may not 
be the crown jewel of his career, but it was wittier than "Devil Wears Prad
a" and more interesting than "Superman" a great comedy to go see with frien
ds.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
<START>Basically there's a family where a little boy (Jake) thinks there's a zomb
ie in his closet & his parents are fighting all the time.This movie is slow
er than a soap opera... and suddenly, Jake decides to become Rambo and kill
 the zombie.OK, first of all when you're going to make a film you must Deci
de if its a thriller or a drama! As a drama the movie is watchable. Parents
 are divorcing & arguing like in real life. And then we have Jake with his 
closet which totally ruins all the film! I expected to see a BOOGEYMAN simi
lar movie, and instead i watched a drama with some meaningless thriller spo
ts.3 out of 10 just for the well playing parents & descent dialogs. As for 
the shots with Jake: just ignore them.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
----------------
```

Test making overlapping sequences of 2 reviews

In [9]:
# Focus on 3rd and 5th reviews in this test

seq_len = 121
seq_overlap = 3

seqs_x_str, seqs_y_str = imdb_ds.make_sequences(corpus=[corpus[2], corpus[4]], seq_len=seq_len, seq_overlap=seq_overlap)

print('3rd and 5th reviews:')
print('----------------')
for chunk in seqs_x_str:
    chunk = decode_special_tokens(chunk)
    print(chunk)
print('----------------')

3rd and 5th reviews:
----------------
<START>I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and
and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (ev
(even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2
t 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown
own to love.This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never be
 been impressed with Scarlet Johanson, in this she managed to tone down her "sexy" image and jumped right into a average,
ge, but spirited young woman.This may not be the crown jewel of his career, but it was wittier than "Devil Wears Prada" a
" and more interesting than "Superman" a great comedy to go see with friends.<END><PAD><PAD><PAD><PAD>

The output of the cell above should be:

*Note: With the exception of the PAD token, pay attention that your lines break at exactly the correct chars.*

*Note: Pay special attention to the continuity/overlap in successive sequences.*

```
3rd and 5th reviews:
----------------
<START>I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and
and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (ev
(even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2
t 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown
own to love.This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never be
 been impressed with Scarlet Johanson, in this she managed to tone down her "sexy" image and jumped right into a average,
ge, but spirited young woman.This may not be the crown jewel of his career, but it was wittier than "Devil Wears Prada" a
" and more interesting than "Superman" a great comedy to go see with friends.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
<START>Petter Mattei's "Love in the Time of Money" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait 
it about human relations. This is a movie that seems to be telling us what money, power and success do to people in the d
e different situations we encounter. This being a variation on the Arthur Schnitzler's play about the same theme, the dir
director transfers the action to the present time New York where all these different characters meet and connect. Each on
 one is connected in one way, or another to the next person, but no one seems to know the previous point of contact. Styl
tylishly, the film has a sophisticated luxurious look. We are taken to see how these people live and the world they live 
ve in their own habitat.The only thing one gets out of all these souls in the picture is the different stages of loneline
iness each one inhabits. A big city is not exactly the best place in which human relations find sincere fulfillment, as o
s one discerns is the case with most of the people we encounter.The acting is good under Mr. Mattei's direction. Steve Bu
 Buscemi, Rosario Dawson, Carol Kane, Michael Imperioli, Adrian Grenier, and the rest of the talented cast, make these ch
 characters come alive.We wish Mr. Mattei good luck and await anxiously for his next work.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
----------------
```

Test `x` and `y` sequences

In [10]:
# Focus on first 4 sequences of 3rd review in this test

seq_len = 125
seq_overlap = 5

seqs_x_str, seqs_y_str = imdb_ds.make_sequences(corpus=[corpus[2]], seq_len=seq_len, seq_overlap=seq_overlap)

for x_seq, y_seq in zip(seqs_x_str[:4], seqs_y_str[:4]):
    x_seq = decode_special_tokens(x_seq)
    y_seq = decode_special_tokens(y_seq)
    print('****Current x sequence:****')
    print(x_seq)
    print('****Current y sequence:****')
    print(y_seq)
    print(25*'-')

****Current x sequence:****
<START>I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and wat
****Current y sequence:****
I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watc
-------------------------
****Current x sequence:****
d watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the
****Current y sequence:****
 watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the 
-------------------------
****Current x sequence:****
n the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk A
****Current y sequence:****
 the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Ad
------------------

Check each of the 4 sequences and next-char sequences carefully. You should know what to look for 😊.

### 1e. Convert char sequences into int coded sequences

Next, implement the `convert_str2int(self, seqs, char2ind)` method, which converts char sequences to int-coded sequences (either input x or target y sequences). This int-coded version will be used to train the net, so it should be int TensorFlow int32 format.

#### Test: Converting a few char sequences into int sequences

In [11]:
seq_len = 50
seq_overlap = 2

seqs_x_str, seqs_y_str = imdb_ds.make_sequences(corpus=corpus, seq_len=seq_len, seq_overlap=seq_overlap)
seqs_x_ints_tf = imdb_ds.convert_str2int(seqs_x_str, char2ind_map)
print(f'After converting to an int code, the shape is {seqs_x_ints_tf.shape} and should be (118, 50).')
print(f"The {seqs_x_ints_tf.dtype} and should be <dtype: 'int32'>.")
print('Your 1st, 10th, and last int-coded seqs are:')
for i in [0, 10, -1]:
    print('Seq', i, ':', seqs_x_ints_tf[i].numpy())
print('and they should be:')
print('''Seq 0 : [ 1 33 54 45  3 55 46  3 60 48 45  3 55 60 48 45 58  3 58 45 62 49 45 63
 45 58 59  3 48 41 59  3 53 45 54 60 49 55 54 45 44  3 60 48 41 60  3 41
 46 60]
Seq 10 : [ 3 60 48 41 60  3 49 59  3 60 48 45  3 54 49 43 51 54 41 53 45  3 47 49
 62 45 54  3 60 55  3 60 48 45  3 33 59 63 41 52 44  3 31 41 64 49 53 61
 53  3]
Seq -1 : [59 52 65  3 46 55 58  3 48 49 59  3 54 45 64 60  3 63 55 58 51 12  2  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0]
''')

2026-05-08 00:57:12.356395: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-08 00:57:13.064184: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-08 00:57:13.068357: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

After converting to an int code, the shape is (118, 50) and should be (118, 50).
The <dtype: 'int32'> and should be <dtype: 'int32'>.
Your 1st, 10th, and last int-coded seqs are:
Seq 0 : [ 1 33 54 45  3 55 46  3 60 48 45  3 55 60 48 45 58  3 58 45 62 49 45 63
 45 58 59  3 48 41 59  3 53 45 54 60 49 55 54 45 44  3 60 48 41 60  3 41
 46 60]
Seq 10 : [ 3 60 48 41 60  3 49 59  3 60 48 45  3 54 49 43 51 54 41 53 45  3 47 49
 62 45 54  3 60 55  3 60 48 45  3 33 59 63 41 52 44  3 31 41 64 49 53 61
 53  3]
Seq -1 : [59 52 65  3 46 55 58  3 48 49 59  3 54 45 64 60  3 63 55 58 51 12  2  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0]
and they should be:
Seq 0 : [ 1 33 54 45  3 55 46  3 60 48 45  3 55 60 48 45 58  3 58 45 62 49 45 63
 45 58 59  3 48 41 59  3 53 45 54 60 49 55 54 45 44  3 60 48 41 60  3 41
 46 60]
Seq 10 : [ 3 60 48 41 60  3 49 59  3 60 48 45  3 54 49 43 51 54 41 53 45  3 47 49
 62 45 54  3 60 55  3 60 48 45  3 33 59 63 41 52 44  3 31 41 64 49 53

### 1f. Process dataset

Just like in `WordLevelDataset`, combine all the preprocessing steps that you implemented thus far into the single `process` method.

#### Test: Preprocess entire IMDb corpus

Each test should take (much) less than 1 minute. If they take longer, something might be wrong.

Test: 100% Train / no validation

In [12]:
seq_len = 128
seq_overlap = 32
imdb_ds = CharLevelDataset(file_path='data/imdb_raw.csv')
imdb_ds.process(-1, seq_len=seq_len, seq_overlap=seq_overlap, prop_val=0.)

seqs_x_str, seqs_y_str = imdb_ds.get_train_split_str()
seqs_x_int, seqs_y_int = imdb_ds.get_train_split_int()

print(f'Your vocab has {len(imdb_ds.get_vocab())} tokens and it should have 99.')

print()
print('Length of your str coded sequences:')
print(f'{len(seqs_x_str)},  {len(seqs_y_str)}')
print('They should be:')
print('694981,  694981')
print()
print('Shape of your int coded sequences:')
print(f'{seqs_x_int.shape}, {seqs_y_int.shape}')
print('They should be:')
print('(694981, 128), (694981, 128)')

Number of unique chars/tokens: 99


Number of train sequences: 694981 of length 128


  seqs_x_int.shape=TensorShape([694981, 128]) seqs_y_int.shape=TensorShape([694981, 128])
Number of val sequences: 0 of length 128
  seqs_x_int.shape=TensorShape([0]) seqs_y_int.shape=TensorShape([0])
Your vocab has 99 tokens and it should have 99.

Length of your str coded sequences:
694981,  694981
They should be:
694981,  694981

Shape of your int coded sequences:
(694981, 128), (694981, 128)
They should be:
(694981, 128), (694981, 128)


Test: 80% train / 20% validation

In [13]:
seq_len = 128
seq_overlap = 32
imdb_ds = CharLevelDataset(file_path='data/imdb_raw.csv')
imdb_ds.process(-1, seq_len=seq_len, seq_overlap=seq_overlap, prop_val=0.2)

seqs_x_train_str, seqs_y_train_str = imdb_ds.get_train_split_str()
seqs_x_train_int, seqs_y_train_int = imdb_ds.get_train_split_int()

seqs_x_val_str, seqs_y_val_str = imdb_ds.get_val_split_str()
seqs_x_val_int, seqs_y_val_int = imdb_ds.get_val_split_int()

print(f'Your vocab has {len(imdb_ds.get_vocab())} tokens and it should have 99.')

print('-------------TRAINING SET-------------')
print()
print('Length of your str coded sequences:')
print(f'{len(seqs_x_train_str)},  {len(seqs_y_train_str)}')
print('They should be:')
print('555360,  555360')
print()
print('Shape of your int coded sequences:')
print(f'{seqs_x_train_int.shape}, {seqs_y_train_int.shape}')
print('They should be:')
print('555360, 128), (555360, 128)')

print('-------------VALIDATION SET-------------')
print()
print('Length of your str coded sequences:')
print(f'{len(seqs_x_val_str)},  {len(seqs_y_val_str)}')
print('They should be:')
print('139621,  139621')
print()
print('Shape of your int coded sequences:')
print(f'{seqs_x_val_int.shape}, {seqs_y_val_int.shape}')
print('They should be:')
print('(139621, 128), (139621, 128)')

Number of unique chars/tokens: 99


Number of train sequences: 555360 of length 128


  seqs_x_int.shape=TensorShape([555360, 128]) seqs_y_int.shape=TensorShape([555360, 128])


Number of val sequences: 139621 of length 128


  seqs_x_int.shape=TensorShape([139621, 128]) seqs_y_int.shape=TensorShape([139621, 128])
Your vocab has 99 tokens and it should have 99.
-------------TRAINING SET-------------

Length of your str coded sequences:
555360,  555360
They should be:
555360,  555360

Shape of your int coded sequences:
(555360, 128), (555360, 128)
They should be:
555360, 128), (555360, 128)
-------------VALIDATION SET-------------

Length of your str coded sequences:
139621,  139621
They should be:
139621,  139621

Shape of your int coded sequences:
(139621, 128), (139621, 128)
They should be:
(139621, 128), (139621, 128)
